# Task 01: Zero-Shot NER with GLiNER Bi-Encoder

Use `knowledgator/gliner-bi-edge-v2.0` to extract named entities from news articles.

You will:
1. Load the model and extract entities from a single text
2. Implement a batch extraction function
3. Pre-compute label embeddings and use them for efficient inference

## Setup

In [1]:
from gliner import GLiNER
import json, os

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "news_articles.json")) as f:
    articles = json.load(f)

print(f"✓ Loaded {len(articles)} articles")

✓ Loaded 10 articles


## Part 1: Load Model and Extract Entities

Load `knowledgator/gliner-bi-edge-v2.0` and extract entities from `articles[0]["text"]`.
Use labels `["person", "organization", "location", "date"]` with `threshold=0.3`.

Store the model in `model` and the entity list in `entities`.

In [2]:
# YOUR CODE HERE


In [3]:
# TEST — model loaded
assert 'model' in dir() or 'model' in vars(), "Load the model into variable 'model'"
assert hasattr(model, 'predict_entities'), "model must have .predict_entities() method"
print("✓ model loaded")

AssertionError: Load the model into variable 'model'

In [4]:
# TEST — entity format
assert 'entities' in dir() or 'entities' in vars(), "Store results in variable 'entities'"
assert isinstance(entities, list), "predict_entities must return a list"
assert len(entities) > 0, "At least one entity should be found in the first article"

required_keys = {"text", "label", "start", "end", "score"}
for i, e in enumerate(entities):
    missing = required_keys - e.keys()
    assert not missing, f"Entity {i} missing keys: {missing}"

# Verify offsets are correct
text = articles[0]["text"]
for e in entities:
    assert text[e["start"]:e["end"]] == e["text"], (
        f"Offset mismatch: text[{e['start']}:{e['end']}]={text[e['start']:e['end']]!r} "
        f"!= entity['text']={e['text']!r}"
    )

print(f"✓ Found {len(entities)} entities with correct format")
for e in entities:
    print(f"  [{e['label']:12}] {e['text']!r:30} score={e['score']:.2f}")

AssertionError: Store results in variable 'entities'

## Part 2: Batch Extraction Function

Implement `extract_entities(model, texts, labels, threshold=0.3) -> list[list[dict]]`
that processes a list of texts and returns a list of entity lists (one per text).

Hint: use `model.inference()` for efficient batch processing.

In [5]:
# YOUR CODE HERE
def extract_entities(model, texts, labels, threshold=0.3):
    pass


In [6]:
# TEST — batch extraction
labels = ["person", "organization", "location", "date"]
texts = [a["text"] for a in articles]
results = extract_entities(model, texts, labels)

assert isinstance(results, list), "extract_entities must return a list"
assert len(results) == len(articles), f"Expected {len(articles)} results, got {len(results)}"

for i, res in enumerate(results):
    assert isinstance(res, list), f"Result {i} must be a list of entity dicts"

# Count entities per article
counts = [len(r) for r in results]
total = sum(counts)
assert total > 0, "No entities found across all articles — check your implementation"

print(f"✓ Processed {len(results)} articles, {total} total entities")
for i, (article, count) in enumerate(zip(articles, counts)):
    persons = [e['text'] for e in results[i] if e['label'] == 'person']
    print(f"  [{article['domain']:15}] {count} entities  persons={persons}")

NameError: name 'model' is not defined

## Part 3: Pre-Compute Label Embeddings

Pre-compute embeddings for the label set using `model.encode_labels(labels)` and store them
in `entity_embeddings`. Then run inference using `model.batch_predict_with_embeds()`
and verify the results are consistent with direct `.inference()` calls.

Store pre-computed embeddings in `entity_embeddings`.

In [7]:
# YOUR CODE HERE
labels = ["person", "organization", "location", "date"]

# entity_embeddings = ...


In [8]:
# TEST — embeddings shape
assert 'entity_embeddings' in dir() or 'entity_embeddings' in vars(), \
    "Store embeddings in 'entity_embeddings'"
assert entity_embeddings is not None, "entity_embeddings must not be None"
assert hasattr(entity_embeddings, 'shape'), "entity_embeddings must be a tensor/array with .shape"
assert entity_embeddings.shape[0] == len(labels), (
    f"Expected {len(labels)} label embeddings, got {entity_embeddings.shape[0]}"
)
print(f"✓ entity_embeddings shape: {entity_embeddings.shape}")

AssertionError: Store embeddings in 'entity_embeddings'

In [9]:
# YOUR CODE HERE
# Use model.batch_predict_with_embeds() on articles[:5] and store results in results_precomputed
texts_sample = [a["text"] for a in articles[:5]]

# results_precomputed = ...


In [10]:
# TEST — pre-computed results consistency
assert 'results_precomputed' in dir() or 'results_precomputed' in vars(), \
    "Store results in 'results_precomputed'"
assert isinstance(results_precomputed, list), "results_precomputed must be a list"
assert len(results_precomputed) == 5, f"Expected 5 results, got {len(results_precomputed)}"

# Compare with direct run
results_direct = model.inference(texts_sample, labels, threshold=0.3)

pre_counts = [len(r) for r in results_precomputed]
dir_counts = [len(r) for r in results_direct]

print(f"✓ Pre-computed results: {pre_counts}")
print(f"  Direct results:       {dir_counts}")

# Counts should be close (minor differences allowed due to threshold behavior)
total_diff = sum(abs(p - d) for p, d in zip(pre_counts, dir_counts))
assert total_diff <= len(texts_sample), (
    f"Pre-computed and direct results differ too much: {pre_counts} vs {dir_counts}"
)
print("✓ Pre-computed embeddings produce consistent results")

AssertionError: Store results in 'results_precomputed'